# Defence Spending & Redistribution — World Bank / IMF Data Pipeline

Companion project to my LSE paper *"What Political Factors Determine the Extent of
Redistribution in Democratic Societies?"* (Romer–Meltzer–Richard model extended with
Effective Political Voice, Electoral Rules, and Defence Spending).

Pulls **defence spending** and **government expenditure** (World Bank) plus **general
government net lending/borrowing** (IMF Government Finance Statistics), merges, cleans,
and visualizes them with `pandas` / `matplotlib` / `seaborn`.

**Run this on your own machine with normal internet access** (needs to reach
`api.worldbank.org` and the IMF's data API — both free, no key required).

```bash
pip install -r requirements.txt
jupyter notebook wb_imf_defence_spending.ipynb
```


In [ ]:
import wbgapi as wb
import imfp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# --- Edit this list to match the countries in your RMR paper (ISO3 codes) ---
COUNTRIES = ["POL", "DEU", "FRA", "ITA", "SWE", "ESP"]
START_YEAR, END_YEAR = 2000, 2024


## 1. World Bank data (`wbgapi`)

Two World Development Indicators (WDI) series:
- `MS.MIL.XPND.GD.ZS` — Military expenditure (% of GDP)
- `NE.CON.GOVT.ZS` — General government final consumption expenditure (% of GDP)


In [ ]:
wdi_series = {
    "MS.MIL.XPND.GD.ZS": "defence_pct_gdp",
    "NE.CON.GOVT.ZS": "govt_expenditure_pct_gdp",
}

wb_frames = []
for code, label in wdi_series.items():
    df = wb.data.DataFrame(code, COUNTRIES, range(START_YEAR, END_YEAR + 1), labels=False)
    df = df.reset_index().melt(id_vars="economy", var_name="year", value_name=label)
    df["year"] = df["year"].str.replace("YR", "").astype(int)
    df = df.rename(columns={"economy": "country"})
    wb_frames.append(df.set_index(["country", "year"]))

wb_data = pd.concat(wb_frames, axis=1).reset_index()
wb_data.head()


## 2. IMF data (`imfp`) — Government Finance Statistics

The IMF's dataset IDs and indicator codes aren't something you hardcode blindly — they're
looked up live from the API, which is exactly what the four cells below do automatically:

1. Confirm the `GFS` (Government Finance Statistics) dataflow exists.
2. Pull the codelist for the `INDICATOR` dimension and **search it by keyword** for
   "net lending" — no manual code-hunting needed.
3. Pull the codelist for the `COUNTRY` dimension and keep only our target countries.
4. Fetch the actual series with `imf_get`.

If step 2 finds no match (the IMF occasionally renames things), it prints the full
indicator codelist so you can eyeball the right one and set `INDICATOR_CODE` manually.


In [ ]:
GFS_DATAFLOW = "GFS"

# Step 1: sanity-check the dataflow exists
dataflows = imfp.imf_get_dataflows()
assert (dataflows["id"] == GFS_DATAFLOW).any(), "GFS dataflow id not found — inspect `dataflows` manually"
dataflows[dataflows["id"] == GFS_DATAFLOW]


In [ ]:
# Step 2: find the indicator code for "net lending/borrowing" automatically
indicator_codes = imfp.imf_get_codelists(["INDICATOR"], GFS_DATAFLOW)
match = indicator_codes[indicator_codes["name"].str.contains("net lending", case=False, na=False)]

if match.empty:
    print("No automatic match — inspect the full list below and set INDICATOR_CODE manually:")
    display(indicator_codes)
    INDICATOR_CODE = None
else:
    INDICATOR_CODE = match.iloc[0]["code"]
    print(f"Using indicator code: {INDICATOR_CODE} ({match.iloc[0]['name']})")


In [ ]:
# Step 3: find the matching country codes
country_codes = imfp.imf_get_codelists(["COUNTRY"], GFS_DATAFLOW)
our_country_codes = country_codes[country_codes["code"].isin(COUNTRIES)]["code"].tolist()

if len(our_country_codes) < len(COUNTRIES):
    missing = set(COUNTRIES) - set(our_country_codes)
    print(f"Note: {missing} not found under these exact ISO3 codes in the GFS country codelist — "
          f"inspect `country_codes` to find the right ones.")

our_country_codes


In [ ]:
# Step 4: fetch the actual series
imf_raw = imfp.imf_get(
    GFS_DATAFLOW,
    dimensions={"COUNTRY": our_country_codes, "INDICATOR": INDICATOR_CODE},
    start_period=START_YEAR,
    end_period=END_YEAR,
)

imf_data = imf_raw.rename(columns={
    "COUNTRY": "country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "net_lending_pct_gdp",
})[["country", "year", "net_lending_pct_gdp"]]
imf_data["year"] = imf_data["year"].astype(int)
imf_data["net_lending_pct_gdp"] = imf_data["net_lending_pct_gdp"].astype(float)
imf_data.head()


## 3. Clean & merge

In [ ]:
merged = wb_data.merge(imf_data, on=["country", "year"], how="left")
merged = merged.dropna(subset=["defence_pct_gdp"])
merged.head()


## 4. Visualize

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=merged, x="year", y="defence_pct_gdp", hue="country", marker="o", ax=ax)
ax.set_title("Defence Spending (% of GDP) by Country")
ax.set_xlabel("Year")
ax.set_ylabel("Defence spending (% of GDP)")
plt.tight_layout()
plt.savefig("defence_spending.png", dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=merged, x="govt_expenditure_pct_gdp", y="defence_pct_gdp",
    hue="country", size="net_lending_pct_gdp", sizes=(30, 250), ax=ax,
)
ax.set_title("Government Expenditure vs. Defence Spending (bubble size = fiscal balance)")
ax.set_xlabel("Government expenditure (% of GDP)")
ax.set_ylabel("Defence spending (% of GDP)")
plt.tight_layout()
plt.savefig("expenditure_vs_defence.png", dpi=150)
plt.show()


## 5. Observation

*Replace this with your own reading of the actual output* — e.g. whether higher overall
government expenditure (a proxy for redistributive capacity, per the RMR framework) tends
to move together with or trade off against defence spending across your country set, and
how fiscal balance (net lending/borrowing) relates to that trade-off. Does the pattern
differ between older and newer democracies or different electoral systems, echoing the
extension you made to the RMR model in your LSE paper?
